In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

# T5 generative model (multi-seed)

`t5-base` fine-tuned as binary **text-to-text** sentiment classification, replicating the
LongEval 2023 Task 2 winning entry of Medina-Alias and Şimşek (2023). Seeds 42, 1, 2.

Their setup (from the paper): T5, Adafactor optimiser, learning rate 1e-4, max sequence
length 128, and the checkpoint selected by loss on the 2016 within-time development set.
That matches this project's val-loss early-stopping convention, so it is reused here.

GPT-3, their other model, needs the paid OpenAI API and is out of scope. T5 was their
winning model, so it is the reproducible version of the winning strategy.

Prediction is done by scoring the two label words (`positive` vs `negative`) under the
fine-tuned model and taking the higher-likelihood one. This is more robust than free
generation for a binary task and, as a by-product, gives a positive-class probability that
the threshold and AUROC analyses need.

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score
import torch
from transformers import (
    AutoTokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback, Adafactor, set_seed,
)
from datasets import Dataset
from huggingface_hub import snapshot_download

DATA_DIR   = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))
MODEL_NAME = 't5-base'
SEEDS      = [42, 1, 2]
STRATEGY   = 't5-generative'
MAX_IN     = 128
MAX_TGT    = 4
LABELS     = ['negative', 'positive']   # target strings
PREFIX     = 'sentiment: '
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name())

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda
GPU: Tesla T4


## Data — load once

In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

splits = {
    'train':       load_split(DATA_DIR / 'train_eval/train.json',            label_col='distant_label'),
    'eval':        load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label'),
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}
for name, df in splits.items():
    print(f'{name}: {len(df)} rows')
print('Labels present:', sorted(splits['train']['label'].unique()))

train: 49608 rows
eval: 1344 rows
test_within: 908 rows
test_short: 908 rows
test_long: 908 rows
Labels present: ['negative', 'positive']


## Text-to-text formatting and tokenisation

Each tweet becomes the input `sentiment: <tweet>` and the target is the label word
`positive` or `negative`. Padding is handled per batch by the collator, which also sets
label padding to -100 so it is ignored in the loss.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    inputs  = [PREFIX + t for t in df['pp_text'].tolist()]
    targets = df['label'].tolist()
    d = Dataset.from_dict({'input_text': inputs, 'target_text': targets})
    def tok(batch):
        model_in = tokenizer(batch['input_text'], truncation=True, max_length=MAX_IN)
        labels   = tokenizer(text_target=batch['target_text'], truncation=True, max_length=MAX_TGT)
        model_in['labels'] = labels['input_ids']
        return model_in
    return d.map(tok, batched=True, remove_columns=['input_text', 'target_text'])

train_ds = make_dataset(splits['train'])
eval_ds  = make_dataset(splits['eval'])
print('Train:', len(train_ds), ' Eval:', len(eval_ds))

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/49608 [00:00<?, ? examples/s]

Map:   0%|          | 0/1344 [00:00<?, ? examples/s]

Train: 49608  Eval: 1344


## Positive-class scoring

For each tweet, sum the token log-likelihood of generating `positive` and of generating
`negative`, then softmax the two to get P(positive). The predicted label is the
higher-likelihood word. Both words tokenise to the same length, so the comparison is fair.

In [5]:
@torch.no_grad()
def positive_probs(model, texts, batch_size=64):
    model.eval()
    cand_ids = {lab: tokenizer(lab, return_tensors='pt').input_ids.to(DEVICE) for lab in LABELS}
    out_probs = []
    for i in range(0, len(texts), batch_size):
        batch = [PREFIX + t for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, truncation=True, max_length=MAX_IN,
                        padding=True, return_tensors='pt').to(DEVICE)
        b = enc['input_ids'].size(0)
        lp = {}
        for lab, lab_ids in cand_ids.items():
            labels = lab_ids.repeat(b, 1)
            logits = model(input_ids=enc['input_ids'],
                           attention_mask=enc['attention_mask'], labels=labels).logits
            logp   = torch.log_softmax(logits, dim=-1)
            tok_lp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)      # [b, T]
            mask   = (labels != tokenizer.pad_token_id).float()
            lp[lab] = (tok_lp * mask).sum(dim=1)                            # [b]
        stack = torch.stack([lp['negative'], lp['positive']], dim=1)       # [b, 2]
        out_probs.append(torch.softmax(stack, dim=1)[:, 1].float().cpu().numpy())
    return np.concatenate(out_probs)

## Training loop (seeds 42, 1, 2)

In [6]:
all_results  = {}
per_tweet    = {}
saved_models = {}

collator = DataCollatorForSeq2Seq(tokenizer, model=None, label_pad_token_id=-100)

for SEED in SEEDS:
    print(f'\n{"="*50}  SEED {SEED}')
    set_seed(SEED)
    out_dir = Path(f'/tmp/{STRATEGY}_seed{SEED}')
    out_dir.mkdir(parents=True, exist_ok=True)

    model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

    # Adafactor with a fixed learning rate, following Medina-Alias and Simsek (2023).
    optimizer = Adafactor(model.parameters(), lr=1e-4, scale_parameter=False,
                          relative_step=False, warmup_init=False)

    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), num_train_epochs=25,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        logging_steps=100, fp16=False, seed=SEED, report_to='none',
        predict_with_generate=False)

    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=collator, processing_class=tokenizer,
        optimizers=(optimizer, None),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()

    seed_results = {}
    per_tweet[SEED] = {}
    for name in ['test_within', 'test_short', 'test_long']:
        texts = splits[name]['pp_text'].tolist()
        gold  = (splits[name]['label'] == 'positive').astype(int).to_numpy()
        p_pos = positive_probs(trainer.model, texts)
        pred  = (p_pos >= 0.5).astype(int)
        f1 = f1_score(gold, pred, average='macro')
        seed_results[name] = f1
        per_tweet[SEED][name] = {'p_pos': p_pos, 'pred': pred, 'gold': gold}
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = trainer.model


==================================================  SEED 42


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


  test_within: F1=0.7466
  test_short: F1=0.6839
  test_long: F1=0.6888


## Summary (per-seed)

In [7]:
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short'] - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']  - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")
print()
for sp in ['test_within', 'test_short', 'test_long']:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals, ddof=1):.4f}  [{min(vals):.4f}-{max(vals):.4f}]")

    seed   within    short     long   RPD_short   RPD_long
      42   0.7382   0.6685   0.6854     -0.0945    -0.0715
       1   0.7477   0.6691   0.6883     -0.1051    -0.0794
       2   0.7466   0.6839   0.6888     -0.0840    -0.0774

test_within: mean=0.7442  std=0.0052  [0.7382-0.7477]
test_short: mean=0.6738  std=0.0087  [0.6685-0.6839]
test_long: mean=0.6875  std=0.0018  [0.6854-0.6888]


## Soft-voting macro-F1 (paper convention)

The paper aggregates the three seeds by averaging the positive-class probability and
predicting positive when the mean is at least 0.5, so compute it here to make the
number directly comparable to the other strategies.

In [8]:
for name in ['test_within', 'test_short', 'test_long']:
    gold   = per_tweet[SEEDS[0]][name]['gold']
    p_mean = np.mean([per_tweet[s][name]['p_pos'] for s in SEEDS], axis=0)   # [N]
    pred   = (p_mean >= 0.5).astype(int)
    f1     = f1_score(gold, pred, average='macro')
    print(f'{name}: soft-voting macro-F1 = {f1:.4f}')

test_within: soft-voting macro-F1 = 0.7479
test_short: soft-voting macro-F1 = 0.6806
test_long: soft-voting macro-F1 = 0.6918


## Save per-tweet predictions

One row per test tweet, with the mean positive-class probability across seeds and the
per-seed prediction and probability, so the threshold, AUROC and per-tweet analyses can
use T5 later.

In [9]:
rows = []
for name in ['test_within', 'test_short', 'test_long']:
    ref   = per_tweet[SEEDS[0]][name]
    n     = len(ref['gold'])
    p_mean = np.mean([per_tweet[s][name]['p_pos'] for s in SEEDS], axis=0)
    for i in range(n):
        row = {'split': name.replace('test_', ''),
               'gold': 'positive' if ref['gold'][i] == 1 else 'negative',
               't5_p_pos_mean': float(p_mean[i])}
        for s in SEEDS:
            row[f't5_pred_seed{s}']  = 'positive' if per_tweet[s][name]['pred'][i] == 1 else 'negative'
            row[f't5_p_pos_seed{s}'] = float(per_tweet[s][name]['p_pos'][i])
        rows.append(row)
pred_df = pd.DataFrame(rows)
pred_df.to_csv('t5_generative_predictions.csv', index=False)
print('Saved t5_generative_predictions.csv', pred_df.shape)
pred_df.head()

Saved t5_generative_predictions.csv (2724, 9)


,split,gold,t5_p_pos_mean,t5_pred_seed42,t5_p_pos_seed42,t5_pred_seed1,t5_p_pos_seed1,t5_pred_seed2,t5_p_pos_seed2
0,within,positive,0.923326,positive,0.971847,positive,0.937481,positive,0.860650
1,within,positive,0.404310,negative,0.328793,positive,0.572631,negative,0.311504
2,within,positive,0.979534,positive,0.983398,positive,0.981997,positive,0.973207
3,within,positive,0.909387,positive,0.927439,positive,0.923797,positive,0.876925
4,within,positive,0.041606,negative,0.043238,negative,0.063151,negative,0.018429


## Save models to HuggingFace

In [10]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-t5-generative-seed42


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-t5-generative-seed2
